<a href="https://colab.research.google.com/github/RBorczuch/PlatesOCR/blob/main/PlatesOCR_video.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PlatesOCR video: YOLO plate detector + plate reader

The OCR model from `PlatesOCR.ipynb` reads plates that are already cut out. This notebook adds the missing step for real use, finding the plates, and runs the whole system on photos and on a video:

1. **Plate reader**: rebuild the OCR model and load `final_model.h5`.
2. **Detector dataset**: convert the boxes of the Katowice dataset to YOLO format, with the same training/validation split as the OCR model.
3. **Detector**: train a YOLO11n plate detector (or load `plate_detector.pt`).
4. **Photo test**: detect and read the plates on the validation photos, which neither model has seen, and compare with the annotations.
5. **Video**: track every plate across frames, read it in every frame and combine the readings with a vote. The result is an annotated video, a CSV list of plates and a gallery of the best crops.

Run it in Google Colab with a GPU runtime. See `README.md` for a description of how it works.

## 0. Setup

In [2]:
import math
import re
import shutil
import subprocess
import sys
import time
import xml.etree.ElementTree as ET
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

# TensorFlow (plate reader) and PyTorch (YOLO) share the GPU: let TensorFlow take memory only as needed
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

try:
    from ultralytics import YOLO
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ultralytics"])
    from ultralytics import YOLO

IN_COLAB = "google.colab" in sys.modules
BASE_DIR = Path("/content") if IN_COLAB else Path.cwd()

# Switches
TRAIN_DETECTOR = True   # train the YOLO detector; False loads DETECTOR_PATH (downloaded from GitHub if missing)

# Paths
DATASET_DIR = BASE_DIR / "dataset"                 # Katowice dataset (shared with PlatesOCR.ipynb)
YOLO_DIR = BASE_DIR / "plates_yolo"                # detector training data in YOLO format
OCR_MODEL_PATH = BASE_DIR / "final_model.h5"       # plate reader trained in PlatesOCR.ipynb
DETECTOR_PATH = BASE_DIR / "plate_detector.pt"     # trained YOLO detector
OUTPUT_DIR = BASE_DIR / "video_output"

DATASET_URL = "https://www.kaggle.com/api/v1/datasets/download/piotrstefaskiue/poland-vehicle-license-plate-dataset"
REPO_RAW_URL = "https://github.com/RBorczuch/PlatesOCR/raw/main"

# Video: put your file at VIDEO_PATH, set VIDEO_URL, or upload it when asked (Colab)
VIDEO_PATH = BASE_DIR / "input.mp4"
VIDEO_URL = ""
FRAME_STEP = 1             # process every n-th frame (2-3 makes long 30 fps videos faster)
OUTPUT_MAX_WIDTH = 1280    # the annotated video is scaled down to this width

# Plate reader: must match PlatesOCR.ipynb
CHARACTERS = list("0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ")
PLATE_WIDTH, PLATE_HEIGHT = 400, 100
IMG_WIDTH, IMG_HEIGHT = 200, 50
BOX_MARGIN = 0.06
TRAIN_FRACTION = 0.8
SEED = 42

# Detector
DETECTOR_BASE = "yolo11n.pt"    # small COCO-pretrained YOLO, fine-tuned on plates
DETECTOR_EPOCHS = 80
DETECTOR_IMGSZ = 960            # plates are small in the photos, so a larger input size than the default 640
YOLO_MAX_SIDE = 1920            # training photos are stored downscaled to this size
DETECTION_CONF = 0.4

# Reading votes in the video
MIN_PLATE_LENGTH = 4            # shorter readings are ignored
MIN_READING_CONFIDENCE = 0.5    # readings whose least certain character is below this are ignored
MIN_TRACK_READINGS = 3          # plates seen in fewer frames are left out of the summary

print("TensorFlow", tf.__version__, "| GPUs:", len(tf.config.list_physical_devices("GPU")))

TensorFlow 2.21.0 | GPUs: 1


## 1. Plate reader

The OCR model is rebuilt here with the same architecture as in `PlatesOCR.ipynb`, without the label input and the CTC loss, which are only needed for training. The weights come from `final_model.h5` (downloaded from GitHub if it is not in the working folder).

`read_plates` returns the text and a confidence for each crop. The confidence is the probability of the least certain character, so one unreadable character is enough to make it low.

In [3]:
def download(url, path):
    """Download `url` to `path`."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(path, "wb") as f:
            for chunk in response.iter_content(chunk_size=1 << 20):
                f.write(chunk)


char_to_num = layers.StringLookup(vocabulary=CHARACTERS, mask_token=None)
num_to_char = layers.StringLookup(vocabulary=char_to_num.get_vocabulary(), mask_token=None, invert=True)
NUM_CLASSES = len(char_to_num.get_vocabulary()) + 1  # + CTC blank (the last class)


def conv_block(x, filters, pool, name):
    """Conv 3x3 -> batch normalization -> ReLU, optionally followed by max pooling."""
    x = layers.Conv2D(filters, (3, 3), padding="same", use_bias=False, kernel_initializer="he_normal", name=f"{name}_conv")(x)
    x = layers.BatchNormalization(name=f"{name}_bn")(x)
    x = layers.Activation("relu", name=f"{name}_relu")(x)
    if pool:
        x = layers.MaxPooling2D(pool, name=f"{name}_pool")(x)
    return x


def build_plate_reader():
    """The prediction part of the PlatesOCR.ipynb model: image -> per-step character probabilities."""
    input_img = layers.Input(shape=(IMG_WIDTH, IMG_HEIGHT, 1), name="image", dtype="float32")
    x = conv_block(input_img, 32, (2, 2), "block1")
    x = conv_block(x, 64, (2, 2), "block2")
    x = conv_block(x, 128, None, "block3")
    x = conv_block(x, 128, (1, 2), "block4")
    x = layers.Reshape(target_shape=(IMG_WIDTH // 4, (IMG_HEIGHT // 8) * 128), name="reshape")(x)
    x = layers.Dense(128, activation="relu", name="dense1")(x)
    x = layers.Dropout(0.25)(x)
    x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=0.25))(x)
    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True, dropout=0.25))(x)
    x = layers.Dense(NUM_CLASSES, activation="softmax", name="dense2")(x)
    return keras.models.Model(input_img, x, name="plate_reader")


def to_model_input(image):
    """uint8 grayscale image (H, W, 1) -> float32 tensor (IMG_WIDTH, IMG_HEIGHT, 1), as in training."""
    image = tf.image.convert_image_dtype(image, tf.float32)
    image = tf.image.resize(image, [IMG_HEIGHT, IMG_WIDTH])
    image = (image - tf.reduce_min(image)) / (tf.reduce_max(image) - tf.reduce_min(image) + 1e-6)
    return tf.transpose(image, perm=[1, 0, 2])


def decode_predictions(pred):
    """Greedy CTC decoding: best class per time step, merge repeats, drop blanks."""
    pred = tf.convert_to_tensor(pred)
    sequence_length = tf.fill([tf.shape(pred)[0]], tf.shape(pred)[1])
    decoded, _ = tf.nn.ctc_greedy_decoder(
        tf.transpose(tf.math.log(pred + 1e-7), perm=[1, 0, 2]),
        sequence_length=sequence_length,
    )
    results = tf.sparse.to_dense(decoded[0], default_value=-1)
    texts = []
    for r in results:
        r = tf.boolean_mask(r, r > 0)
        texts.append(tf.strings.reduce_join(num_to_char(r)).numpy().decode("utf-8"))
    return texts


def reading_confidence(probs):
    """Probability of the least certain character: min over the steps that output a character."""
    best, classes = probs.max(axis=1), probs.argmax(axis=1)
    characters = classes != probs.shape[1] - 1  # not the blank
    return float(best[characters].min()) if characters.any() else 0.0


def read_plates(crops):
    """BGR plate crops -> list of (text, confidence)."""
    if not crops:
        return []
    batch = tf.stack([to_model_input(cv2.cvtColor(c, cv2.COLOR_BGR2GRAY)[..., np.newaxis]) for c in crops])
    probs = plate_reader(batch, training=False).numpy()
    return list(zip(decode_predictions(probs), (reading_confidence(p) for p in probs)))


if not OCR_MODEL_PATH.exists():
    download(f"{REPO_RAW_URL}/final_model.h5", OCR_MODEL_PATH)
plate_reader = build_plate_reader()
try:
    plate_reader.load_weights(str(OCR_MODEL_PATH))
except ValueError as e:
    raise RuntimeError(
        f"{OCR_MODEL_PATH} does not fit the plate reader. It must be trained by the current PlatesOCR.ipynb "
        "(same CHARACTERS and architecture)."
    ) from e
print(f"Plate reader loaded from {OCR_MODEL_PATH}")

Plate reader loaded from /content/final_model.h5


## 2. Detector dataset

The YOLO training data comes from the same `annotations.xml` as the plate crops. For each box:

- the four corners are computed (with the rotation, if any) and their axis-aligned bounding rectangle becomes the YOLO box: `class x_center y_center width height`, all relative to the image size,
- the photo is stored downscaled to at most 1920 px, which is enough for the detector and makes training faster (relative coordinates do not change).

**Same split as the OCR model.** The plate numbers are split with the same function and seed as in `PlatesOCR.ipynb`, and every photo goes to the subset of its plate. The validation photos are therefore new to both the detector and the plate reader, which makes the photo test in section 4 a fair test of the whole system.

In [4]:
def download_and_extract(url, target_dir):
    """Download a zip archive from `url` and unpack it into `target_dir`."""
    archive = target_dir / "download.zip"
    download(url, archive)
    with zipfile.ZipFile(archive) as zf:
        zf.extractall(target_dir)
    archive.unlink()


def parse_cvat(xml_path):
    """Yield (photo name, plate number, box element) for every plate in a CVAT 1.1 XML file."""
    root = ET.parse(xml_path).getroot()
    for image in root.iter("image"):
        for box in image.iter("box"):
            text = next((a.text for a in box.iter("attribute") if a.get("name") == "plate number"), None)
            yield image.get("name"), text or "", box


def normalize_plate(text):
    """Plate number as used in file names and labels: no spaces, upper case."""
    return re.sub(r"\s+", "", text).upper()


def box_corners(box, margin=BOX_MARGIN):
    """Corners (TL, TR, BR, BL) of a CVAT box, rotated and enlarged by `margin` x height on every side."""
    xtl, ytl, xbr, ybr = (float(box.get(k)) for k in ("xtl", "ytl", "xbr", "ybr"))
    cx, cy = (xtl + xbr) / 2, (ytl + ybr) / 2
    half_h = (ybr - ytl) / 2
    half_w = (xbr - xtl) / 2 + 2 * margin * half_h
    half_h *= 1 + 2 * margin
    angle = math.radians(float(box.get("rotation", 0)))
    cos, sin = math.cos(angle), math.sin(angle)
    offsets = [(-half_w, -half_h), (half_w, -half_h), (half_w, half_h), (-half_w, half_h)]
    return np.float32([[cx + dx * cos - dy * sin, cy + dx * sin + dy * cos] for dx, dy in offsets])


def box_xyxy(box, width, height):
    """Axis-aligned bounding rectangle (x1, y1, x2, y2) of a CVAT box, clipped to the image."""
    corners = box_corners(box, margin=0)
    x1, y1 = np.clip(corners.min(axis=0), 0, [width, height])
    x2, y2 = np.clip(corners.max(axis=0), 0, [width, height])
    return float(x1), float(y1), float(x2), float(y2)


def validation_plates(plate_numbers, train_fraction=TRAIN_FRACTION, seed=SEED):
    """Validation plate numbers of split_by_plate() in PlatesOCR.ipynb (same sort, seed and fraction)."""
    plates = sorted(set(plate_numbers))
    np.random.default_rng(seed).shuffle(plates)
    return set(plates[round(len(plates) * train_fraction):])


def load_annotations(dataset_dir):
    """{photo path: [(plate number, box element), ...]} for every annotated photo."""
    xml_path = next(dataset_dir.rglob("annotations.xml"))
    photos = {p.name: p for p in dataset_dir.rglob("*.jpg")}
    annotations = defaultdict(list)
    for name, text, box in parse_cvat(xml_path):
        if name in photos and normalize_plate(text):
            annotations[photos[name]].append((normalize_plate(text), box))
    return dict(annotations)


def build_yolo_dataset(annotations, valid_plates, output_dir, max_side=YOLO_MAX_SIDE):
    """Write images/{train,val} and labels/{train,val} in YOLO format, plus data.yaml."""
    shutil.rmtree(output_dir, ignore_errors=True)
    counts = Counter()
    for photo, plates in annotations.items():
        subset = "val" if any(text in valid_plates for text, _ in plates) else "train"
        image = cv2.imread(str(photo))
        if image is None:
            print(f"Cannot read {photo}")
            continue
        height, width = image.shape[:2]
        scale = min(1.0, max_side / max(height, width))
        if scale < 1:
            image = cv2.resize(image, (round(width * scale), round(height * scale)), interpolation=cv2.INTER_AREA)
        for kind in ("images", "labels"):
            (output_dir / kind / subset).mkdir(parents=True, exist_ok=True)
        cv2.imwrite(str(output_dir / "images" / subset / photo.name), image, [cv2.IMWRITE_JPEG_QUALITY, 95])
        lines = []
        for _, box in plates:
            x1, y1, x2, y2 = box_xyxy(box, width, height)
            lines.append(f"0 {(x1 + x2) / 2 / width:.6f} {(y1 + y2) / 2 / height:.6f} "
                         f"{(x2 - x1) / width:.6f} {(y2 - y1) / height:.6f}")
        (output_dir / "labels" / subset / f"{photo.stem}.txt").write_text("\n".join(lines) + "\n")
        counts[subset] += 1
    (output_dir / "data.yaml").write_text(
        f"path: {output_dir.as_posix()}\ntrain: images/train\nval: images/val\nnames:\n  0: plate\n"
    )
    return counts


if not any(DATASET_DIR.rglob("annotations.xml")):
    print("Downloading the dataset...")
    download_and_extract(DATASET_URL, DATASET_DIR)

annotations = load_annotations(DATASET_DIR)
valid_plates = validation_plates(text for plates in annotations.values() for text, _ in plates)
valid_photos = [p for p, plates in annotations.items() if any(text in valid_plates for text, _ in plates)]
print(f"{len(annotations)} photos, {len(valid_photos)} validation photos ({len(valid_plates)} plate numbers)")

if TRAIN_DETECTOR:
    counts = build_yolo_dataset(annotations, valid_plates, YOLO_DIR)
    print(f"YOLO dataset: {counts['train']} training and {counts['val']} validation photos in {YOLO_DIR}")

290 photos, 59 validation photos (53 plate numbers)
YOLO dataset: 231 training and 59 validation photos in /content/plates_yolo


## 3. Detector

YOLO11n is the smallest model of the YOLO11 family (about 2.6 million parameters), pretrained on COCO. Fine-tuning it on one class, `plate`, needs only a few hundred photos. Ultralytics adds its own augmentation (mosaic, scaling, colour changes, flips) during training.

The detector is evaluated on the validation photos with the usual detection measures: **mAP50** (average precision when a box counts as correct if it overlaps the true box by at least 50%) and **mAP50-95** (the same averaged over stricter overlaps, from 50% to 95%). The best weights are copied to `plate_detector.pt`.

With `TRAIN_DETECTOR = False`, `plate_detector.pt` is loaded instead (and downloaded from the GitHub repository, if it was committed there).

In [ ]:
if TRAIN_DETECTOR:
    trainer_model = YOLO(DETECTOR_BASE)
    trainer_model.train(
        data=str(YOLO_DIR / "data.yaml"),
        epochs=DETECTOR_EPOCHS,
        imgsz=DETECTOR_IMGSZ,
        batch=16,
        patience=20,
        seed=SEED,
        project=str(BASE_DIR / "runs"),
        name="plate_detector",
        exist_ok=True,
    )
    shutil.copy(Path(trainer_model.trainer.save_dir) / "weights" / "best.pt", DETECTOR_PATH)
    print(f"Detector saved to {DETECTOR_PATH}")
elif not DETECTOR_PATH.exists():
    download(f"{REPO_RAW_URL}/plate_detector.pt", DETECTOR_PATH)

detector = YOLO(str(DETECTOR_PATH))
if YOLO_DIR.is_dir():
    metrics = detector.val(data=str(YOLO_DIR / "data.yaml"), imgsz=DETECTOR_IMGSZ, split="val", verbose=False)
    print(f"Detector on validation photos: mAP50 = {metrics.box.map50:.3f}, mAP50-95 = {metrics.box.map:.3f}")

Ultralytics 8.4.175 🚀 Python-3.13.15 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/plates_yolo/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=960, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=plate_detector, nbs=64, nms=None, opset=None, optimize=False

## 4. Photo test: detector + plate reader

Every validation photo goes through the complete system: the detector finds the plates, every box is enlarged by the same margin as the training crops, warped to `400 x 100` and read by the plate reader. The results are compared with the annotations:

- **detection recall**: share of annotated plates found by the detector (a detection counts if it overlaps the true box by at least 50%),
- **false detections**: boxes that match no annotated plate,
- **end-to-end accuracy**: share of annotated plates that were found *and* read without a mistake,
- **character error rate (CER)** of the readings of the found plates.

In [ ]:
PLATE_CORNERS = np.float32([
    [0, 0],
    [PLATE_WIDTH - 1, 0],
    [PLATE_WIDTH - 1, PLATE_HEIGHT - 1],
    [0, PLATE_HEIGHT - 1],
])


def warp_plate(image, corners):
    """Map the four plate corners onto a flat PLATE_WIDTH x PLATE_HEIGHT image."""
    matrix = cv2.getPerspectiveTransform(corners, PLATE_CORNERS)
    return cv2.warpPerspective(image, matrix, (PLATE_WIDTH, PLATE_HEIGHT))


def crop_detection(image, xyxy, margin=BOX_MARGIN):
    """Plate crop for a detector box, enlarged by `margin` x height on every side like the training crops."""
    x1, y1, x2, y2 = xyxy
    pad = margin * (y2 - y1)
    corners = np.float32([[x1 - pad, y1 - pad], [x2 + pad, y1 - pad], [x2 + pad, y2 + pad], [x1 - pad, y2 + pad]])
    return warp_plate(image, corners)


def detect_plates(image, model=None):
    """Boxes (N x 4, x1 y1 x2 y2) and confidences of the plates found in a BGR image."""
    result = (model or detector).predict(image, imgsz=DETECTOR_IMGSZ, conf=DETECTION_CONF, verbose=False)[0]
    return result.boxes.xyxy.cpu().numpy(), result.boxes.conf.cpu().numpy()


def read_photo(image):
    """Detect and read all plates in a BGR image: list of (box, text, reading confidence)."""
    boxes, _ = detect_plates(image)
    readings = read_plates([crop_detection(image, b) for b in boxes])
    return [(b, text, conf) for b, (text, conf) in zip(boxes, readings)]


def iou(a, b):
    """Intersection over union of two boxes (x1, y1, x2, y2)."""
    w = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    h = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = w * h
    union = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / union if union > 0 else 0.0


def edit_distance(a, b):
    """Levenshtein distance between two strings."""
    row = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        prev, row[0] = row[0], i
        for j, cb in enumerate(b, 1):
            current = min(row[j] + 1, row[j - 1] + 1, prev + (ca != cb))
            prev, row[j] = row[j], current
    return row[-1]


def evaluate_photos(photos):
    """Run the whole system on annotated photos and compare with the annotations."""
    stats = Counter()
    examples = []
    for photo in photos:
        image = cv2.imread(str(photo))
        height, width = image.shape[:2]
        found = read_photo(image)
        matched = set()
        for text, box in annotations[photo]:
            truth_box = box_xyxy(box, width, height)
            stats["plates"] += 1
            scores = [iou(truth_box, b) for b, _, _ in found]
            best = int(np.argmax(scores)) if scores else -1
            if best < 0 or scores[best] < 0.5:
                examples.append((photo, text, None))
                continue
            matched.add(best)
            reading = found[best][1]
            stats["detected"] += 1
            stats["correct"] += reading == text
            stats["errors"] += edit_distance(reading, text)
            stats["characters"] += len(text)
            examples.append((photo, text, reading))
        stats["false_detections"] += len(found) - len(matched)
    return stats, examples


stats, examples = evaluate_photos(valid_photos)
print(f"Detection recall:     {stats['detected']}/{stats['plates']} = {stats['detected'] / stats['plates']:.1%}")
print(f"False detections:     {stats['false_detections']}")
print(f"End-to-end accuracy:  {stats['correct']}/{stats['plates']} = {stats['correct'] / stats['plates']:.1%}")
print(f"Character error rate: {stats['errors'] / max(stats['characters'], 1):.1%} (found plates)")

mistakes = [(p.name, t, r) for p, t, r in examples if r != t]
if mistakes:
    print("\nMistakes (photo, true plate, reading):")
    for name, truth, reading in mistakes:
        print(f"  {name}: {truth} -> {reading or 'not detected'}")

# A few validation photos with the detected boxes and readings
_, ax = plt.subplots(2, 3, figsize=(18, 8))
for a, photo in zip(ax.ravel(), valid_photos[:6]):
    image = cv2.imread(str(photo))
    for box, text, conf in read_photo(image):
        x1, y1, x2, y2 = map(int, box)
        thickness = max(2, image.shape[1] // 500)
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 220, 0), thickness)
        cv2.putText(image, text, (x1, max(0, y1 - 10)), cv2.FONT_HERSHEY_SIMPLEX, image.shape[1] / 1000, (0, 220, 0), thickness)
    a.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    a.set_title(f"{photo.name}: {', '.join(t for t, _ in annotations[photo])}")
for a in ax.ravel():
    a.axis("off")
plt.tight_layout()
plt.show()

## 5. Video

### How the video is processed

For every frame (or every `FRAME_STEP`-th frame):

1. **Detection and tracking.** YOLO finds the plates and the ByteTrack tracker links the boxes from frame to frame, so each car gets a track id.
2. **Reading.** Every box is cut out, straightened to `400 x 100` and read by the plate reader, which also returns a confidence.
3. **Vote.** A single frame can be blurred or partly hidden. Each track collects the readings from all its frames; readings shorter than 4 characters or with confidence below 0.5 are ignored, the others add their confidence to the vote for their text. The text with the highest total is the plate of that track, and the box in the video is labelled with it together with its share of the vote.

### Which video works

The plate reader was trained on Polish one-row plates photographed from a few metres. Good results need:

- Polish plates, front or back, readable by eye in the video,
- plates at least about 100 px wide in the frame (1080p video from a few metres works well),
- little motion blur: daylight, steady camera or slow cars.

Put the video at `VIDEO_PATH` (`/content/input.mp4` in Colab), set `VIDEO_URL` to a direct link, or upload it when the next cell asks.

In [ ]:
if not VIDEO_PATH.exists():
    if VIDEO_URL:
        download(VIDEO_URL, VIDEO_PATH)
    elif IN_COLAB:
        from google.colab import files
        print("Upload a video file:")
        uploaded = files.upload()
        VIDEO_PATH = BASE_DIR / next(iter(uploaded))
    else:
        raise FileNotFoundError(f"Put a video at {VIDEO_PATH} or set VIDEO_URL.")

capture = cv2.VideoCapture(str(VIDEO_PATH))
fps = capture.get(cv2.CAP_PROP_FPS)
frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
size = (int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT)))
capture.release()
print(f"{VIDEO_PATH.name}: {size[0]}x{size[1]}, {fps:.1f} fps, {frame_count} frames ({frame_count / max(fps, 1):.1f} s)")

In [ ]:
class PlateTrack:
    """Readings of one tracked plate, combined by a confidence-weighted vote."""

    def __init__(self, track_id, time_s):
        self.track_id = track_id
        self.first_seen = self.last_seen = time_s
        self.readings = 0
        self.votes = Counter()  # text -> summed confidence
        self.best_crop, self.best_confidence = None, -1.0

    def add(self, text, confidence, crop, time_s):
        self.last_seen = time_s
        self.readings += 1
        if len(text) >= MIN_PLATE_LENGTH and confidence >= MIN_READING_CONFIDENCE:
            self.votes[text] += confidence
        if confidence > self.best_confidence:
            self.best_crop, self.best_confidence = crop, confidence

    @property
    def plate(self):
        """Text with the highest total confidence, or None if no reading was good enough."""
        return self.votes.most_common(1)[0][0] if self.votes else None

    @property
    def agreement(self):
        """Share of the vote that went to the winning text."""
        return self.votes[self.plate] / sum(self.votes.values()) if self.votes else 0.0


def draw_label(frame, box, label, color):
    """Draw a box with a filled label above it."""
    x1, y1, x2, y2 = map(int, box)
    thickness = max(2, frame.shape[1] // 640)
    scale = max(0.6, frame.shape[1] / 1600)
    cv2.rectangle(frame, (x1, y1), (x2, y2), color, thickness)
    (tw, th), base = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, scale, thickness)
    top = max(y1, th + base + 6)
    cv2.rectangle(frame, (x1, top - th - base - 6), (x1 + tw + 6, top), color, -1)
    cv2.putText(frame, label, (x1 + 3, top - base - 3), cv2.FONT_HERSHEY_SIMPLEX, scale, (0, 0, 0), thickness)


def process_video(source, output_path, frame_step=FRAME_STEP):
    """Detect, track and read plates in a video; write an annotated copy and return the tracks."""
    tracker = YOLO(str(DETECTOR_PATH))  # fresh model, so no tracker state is left from an earlier video
    capture = cv2.VideoCapture(str(source))
    fps = capture.get(cv2.CAP_PROP_FPS) or 25.0
    width, height = int(capture.get(cv2.CAP_PROP_FRAME_WIDTH)), int(capture.get(cv2.CAP_PROP_FRAME_HEIGHT))
    scale = min(1.0, OUTPUT_MAX_WIDTH / width)
    out_size = (round(width * scale) // 2 * 2, round(height * scale) // 2 * 2)
    writer = cv2.VideoWriter(str(output_path), cv2.VideoWriter_fourcc(*"mp4v"), fps / frame_step, out_size)

    tracks = {}
    frame_index = processed = 0
    start = time.time()
    while True:
        ok, frame = capture.read()
        if not ok:
            break
        if frame_index % frame_step == 0:
            time_s = frame_index / fps
            result = tracker.track(frame, persist=True, tracker="bytetrack.yaml", imgsz=DETECTOR_IMGSZ,
                                   conf=DETECTION_CONF, verbose=False)[0]
            if result.boxes.id is not None:
                boxes = result.boxes.xyxy.cpu().numpy()
                ids = result.boxes.id.int().cpu().tolist()
                crops = [crop_detection(frame, b) for b in boxes]
                for track_id, box, crop, (text, conf) in zip(ids, boxes, crops, read_plates(crops)):
                    track = tracks.setdefault(track_id, PlateTrack(track_id, time_s))
                    track.add(text, conf, crop, time_s)
                    if track.plate:
                        draw_label(frame, box, f"{track.plate} {track.agreement:.0%}", (0, 220, 0))
                    else:
                        draw_label(frame, box, "reading...", (0, 200, 255))
            writer.write(cv2.resize(frame, out_size, interpolation=cv2.INTER_AREA))
            processed += 1
            if processed % 100 == 0:
                print(f"  {frame_index}/{frame_count} frames, {len(tracks)} plates so far")
        frame_index += 1
    capture.release()
    writer.release()
    elapsed = time.time() - start
    print(f"Processed {processed} frames in {elapsed:.0f} s ({processed / max(elapsed, 1e-6):.1f} frames/s)")
    return tracks


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
raw_video = OUTPUT_DIR / f"{VIDEO_PATH.stem}_annotated_mp4v.mp4"
tracks = process_video(VIDEO_PATH, raw_video)

### Results

- **Annotated video**: the browser cannot play the `mp4v` codec written by OpenCV, so the video is converted to H.264 with `ffmpeg` (available in Colab) and shown below. Download it from the file panel (`video_output/`).
- **Plate list**: one row per tracked plate that was seen in at least `MIN_TRACK_READINGS` frames, with its final reading, the share of the vote and when it was visible. Saved as `video_output/plates.csv`.
- **Gallery**: the crop with the most confident reading of each plate, to check the readings by eye.

In [ ]:
from IPython.display import Video, display

final_video = OUTPUT_DIR / f"{VIDEO_PATH.stem}_annotated.mp4"
if shutil.which("ffmpeg"):
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(raw_video), "-c:v", "libx264",
                    "-pix_fmt", "yuv420p", "-crf", "23", str(final_video)], check=True)
else:
    shutil.copy(raw_video, final_video)
print(f"Annotated video: {final_video}")
if final_video.stat().st_size < 80 * 1024 * 1024:
    display(Video(str(final_video), embed=True, width=960))
else:
    print("The video is too large to show here; download it from the file panel.")

rows = [
    {"track": t.track_id, "plate": t.plate, "vote share": round(t.agreement, 2), "frames": t.readings,
     "first seen [s]": round(t.first_seen, 1), "last seen [s]": round(t.last_seen, 1)}
    for t in tracks.values() if t.plate and t.readings >= MIN_TRACK_READINGS
]
plates_table = pd.DataFrame(rows, columns=["track", "plate", "vote share", "frames", "first seen [s]", "last seen [s]"])
plates_table.to_csv(OUTPUT_DIR / "plates.csv", index=False)
display(plates_table)

gallery = [t for t in tracks.values() if t.plate and t.readings >= MIN_TRACK_READINGS]
if gallery:
    columns = 4
    rows_count = math.ceil(len(gallery) / columns)
    _, ax = plt.subplots(rows_count, columns, figsize=(16, 1.6 * rows_count), squeeze=False)
    for a in ax.ravel():
        a.axis("off")
    for a, t in zip(ax.ravel(), gallery):
        a.imshow(cv2.cvtColor(t.best_crop, cv2.COLOR_BGR2RGB))
        a.set_title(f"#{t.track_id}: {t.plate} ({t.agreement:.0%})")
    plt.tight_layout()
    plt.show()